<a href="https://colab.research.google.com/github/simongalaxy/sentimental_agent/blob/master/Sentimental_analysis_amazon_bedsure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [71]:
# ! pip install ollama

In [72]:
import pandas as pd

In [73]:
filepath = "/content/Google_Locations_Central_City_Fitness_&_CrossFit.csv"

In [74]:
df = pd.read_csv(filepath)

In [75]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139 entries, 0 to 138
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             139 non-null    int64  
 1   Platform               139 non-null    object 
 2   Product/Location_Name  139 non-null    object 
 3   Review_Text            139 non-null    object 
 4   Total_Reviews          139 non-null    int64  
 5   Average_Rating         139 non-null    float64
 6   Total_Helpful_Votes    139 non-null    int64  
 7   Sentiment              139 non-null    object 
 8   Category               139 non-null    object 
dtypes: float64(1), int64(3), object(5)
memory usage: 9.9+ KB


In [76]:
df.head(6)

,Unnamed: 0,Platform,Product/Location_Name,Review_Text,Total_Reviews,Average_Rating,Total_Helpful_Votes,Sentiment,Category
0,0,Google Locations,Central City Fitness & CrossFit,Absolute 5-star experience! Our server was kno...,1,5.0,0,POSITIVE,Customer Service
1,1,Google Locations,Central City Fitness & CrossFit,Absolute 5-star experience! Our server was kno...,2,5.0,10,POSITIVE,Customer Service
2,2,Google Locations,Central City Fitness & CrossFit,Absolute 5-star experience! Our server was kno...,2,5.0,2,POSITIVE,Customer Service
3,3,Google Locations,Central City Fitness & CrossFit,Absolute 5-star experience! Our server was kno...,2,5.0,25,POSITIVE,Customer Service
4,4,Google Locations,Central City Fitness & CrossFit,Absolute 5-star experience! Our server was kno...,3,5.0,3,POSITIVE,Customer Service
5,5,Google Locations,Central City Fitness & CrossFit,Absolute 5-star experience! Our server was kno...,2,5.0,8,POSITIVE,Customer Service


In [77]:
df["Sentiment"] = df["Sentiment"].str.strip()
df["Category"] = df["Category"].str.strip()

In [78]:
from collections import Counter
Counter(df["Category"])

Counter({'Customer Service': 90, 'Battery Life': 29, 'Pricing': 20})

In [79]:
df.groupby(by=["Category", "Sentiment"]).agg(Rows=("Review_Text", "count"), Reviews=("Total_Reviews", "sum"), Average_Rating=("Average_Rating", "mean"), Helpful_Votes=("Total_Helpful_Votes", "sum"))

Rows  Reviews  Average_Rating  Helpful_Votes
Category         Sentiment                                              
Battery Life     NEGATIVE      9       10        1.888889             21
                 POSITIVE     20       29        4.400000             48
Customer Service NEGATIVE     32       37        1.468750            215
                 POSITIVE     58      104        4.603448            452
Pricing          NEGATIVE      8        9        1.250000            107
                 POSITIVE     12       16        3.583333             77

In [80]:
from ollama import Client
from google.colab import userdata

class LLMAgent:
  def __init__(self):
    self.model = "gpt-oss:120b-cloud"
    self.host = "https://ollama.com"
    self.ollama_api_key = userdata.get('ollama_api_key')
    self.client = Client(
        host=self.host,
        headers={'Authorization': self.ollama_api_key}
        )

  def get_insights(self, reviews_markdown: str) -> str:
    # Send it to a local Ollama model
    prompt = f"""
    Summarize the reviews into top 5 points with figure support:

    Review Markdown:
    {reviews_markdown}
    """

    response = self.client.chat(
        model=self.model,
        messages=[{"role": "user", "content": prompt}]
        )

    insights = response['message']['content']

    return insights

In [81]:
# save markdown to markdown file.
import textwrap

def save_wrapped_markdown(filename: str, markdown_content: str):
    """
    Wraps paragraph text inside a Markdown string to a specific width
    and saves it to a file, preserving structural elements like headings and lists.
    """
    wrapped_lines = []

    # Split content into paragraphs/blocks by empty lines
    blocks = markdown_content.split('\n\n')

    for block in blocks:
        # Strip trailing/leading spaces from the block
        block = block.strip()
        if not block:
            continue

        # Do not wrap Markdown structural elements (headings, list items, code blocks)
        if (block.startswith('#') or
            block.startswith('- ') or
            block.startswith('* ') or
            block.startswith('>') or
            block.startswith('```')):
            wrapped_lines.append(block)
        else:
            # Wrap standard paragraphs to the specified column width
            wrapped_paragraph = textwrap.fill(block, width=80)
            wrapped_lines.append(wrapped_paragraph)

    # Rejoin the wrapped blocks with double newlines for proper Markdown formatting
    final_output = '\n\n'.join(wrapped_lines)

    # Save to the .md file
    with open(filename, 'w', encoding='utf-8') as file:
        file.write(final_output)
    print(f"✅ Successfully saved beautifully wrapped Markdown to: {filename}")

    return
# # --- Example Usage ---
# raw_markdown = """# Project Overview

# This is a very long sentence that spans across multiple screens and can be incredibly tiring to read if the line length isn't restricted by an automated paragraph text-wrapping utility script.

# - An item in a list that we don't want to break aggressively.
# - Another structural item.

# This is another standard body paragraph filled with dummy text designed to demonstrate exactly how the textwrap module dynamically breaks down extra long lines into bite-sized, readable chunks without destroying your Markdown format layout configurations."""

# # Save the file with a maximum line width of 80 characters
# save_wrapped_markdown("readable_document.md", raw_markdown, width=80)


In [82]:
Agent = LLMAgent()

In [83]:
for category in df["Category"].unique():
  for sentiment in df["Sentiment"].unique():
    print(f"Category: {category}")
    print(f"Sentiment: {sentiment}")
    views_markdown = df[(df["Category"] == category) & (df["Sentiment"] == sentiment)].sort_values(by="Total_Reviews", ascending=False)[["Review_Text", "Total_Reviews", "Average_Rating", "Total_Helpful_Votes"]].to_markdown(index=False)
    insights = Agent.get_insights(reviews_markdown=views_markdown)
    print(f"Insights: {insights}")
    save_wrapped_markdown(
        filename= f"Insight_{df["Platform"].unique()[0].strip()}-{df["Product/Location_Name"].unique()[0].strip()}.md",
        markdown_content=insights
        )

Category: Customer Service
Sentiment: POSITIVE
Insights: **Top 5 Take‑aways from the 197 review excerpts**  

| # | Core theme (what customers rave about / complain about) | # of individual reviews* | Weighted average rating | Total “helpful” votes** |
|---|--------------------------------------------------------|--------------------------|------------------------|--------------------------|
| 1 | **Exceptional hospitality – “staff treated us like royalty”, gorgeous suites & spa** | **28** | **5.0** | **≈ 129** |
| 2 | **Phenomenal dining – “culinary masterpiece”, impeccable service & magical ambience** | **≈ 33** | **5.0** | **≈ 111** |
| 3 | **World‑class care & professionalism – medical/ wellness staff go “above and beyond”** | **≈ 31** | **5.0** | **≈ 147** |
| 4 | **Outstanding service & honest pricing (auto‑repair) – “found my forever mechanic”** | **≈ 33** | **5.0** | **≈ 64** |
| 5 | **Negative / mixed experiences – cleanliness issues, hidden fees, rude management** | **4** | *